In [ ]:
import json
import os
from typing import Any, List

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely import Polygon
import shapely.geometry as sg
from sklearn.cluster import DBSCAN

from notebooks.dataset_utils import read_annotations_folder

RD_EPSG = 28992  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_EPSG = 4326  # CRS code for WGS84 latitude/longitude coordinate system

MS_PER_RAD: float = 6371008.8  # Earth radius in meters
MIN_SAMPLES: int = 1  # avoid noise points. All points are either in a cluster or are a cluster of their own.


def cluster_and_select_images(gdf: gpd.GeoDataFrame, distance: float = 10.0) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    points = np.array([[p.x, p.y] for p in gdf["geometry"]])

    # Cluster the points based on distance
    if gdf.crs.to_epsg() == LAT_LON_EPSG:
        epsilon = distance / MS_PER_RAD
        points = np.radians(points)
        metric = "haversine"
    elif gdf.crs.to_epsg() == RD_EPSG:
        epsilon = distance
        metric = "euclidean"
    
    db = DBSCAN(
        eps=epsilon, min_samples=MIN_SAMPLES, algorithm="ball_tree", metric=metric
    ).fit(points)

    gdf["tracking_id"] = db.labels_

    # # For each tracking_id group, discard rows with 'confidence' value below the average of the group
    # conf_select = df["confidence"] >= df.groupby("tracking_id")["confidence"].transform(
    #     "mean"
    # )

    # # Group by cluster and select the image with the largest area
    # df["area"] = df["width"] * df["height"]
    # df["selected"] = False
    # df.iloc[df[conf_select].groupby("tracking_id")["area"].idxmax().values, -1] = True

    return gdf

def read_buurten_geojson(buurten_file: str) -> gpd.GeoDataFrame:
    data: List[dict[str, Any]] = []

    with open(buurten_file, "r") as f:
        json_content = json.load(f)

        for buurt in json_content["features"]:
            data.append(
                {
                    "name": buurt["properties"]["naam"],
                    "code": buurt["properties"]["code"],
                    "geometry": Polygon(buurt["geometry"]["coordinates"][0]),
                }
            )
    return gpd.GeoDataFrame(data=data).set_crs(epsg=LAT_LON_EPSG)

def add_neighborhoods_to_plot(axes: List[plt.Axes], buurten_gdf: gpd.GeoDataFrame):
    xmin, xmax = axes[0].get_xbound()
    ymin, ymax = axes[0].get_ybound()
    bounds = sg.box(xmin, ymin, xmax, ymax)

    for ax in axes:
        buurten_gdf[buurten_gdf.intersects(bounds)].boundary.plot(ax=ax, color="lightgrey", zorder=1)
        ax.set_xbound((xmin, xmax))
        ax.set_ybound((ymin, ymax))

In [ ]:
dataset_folder = "../datasets/oor/testride_velotech"
coordinates_metadata_file = os.path.join(dataset_folder, "latest_track.json")
annotations_clustered_file = os.path.join(dataset_folder, "clusters_checked_260817.csv")
detections_folder = os.path.join(dataset_folder, "inference/recording_2025-05-14_19-47-40")
detections_validated_folder = os.path.join(dataset_folder, "detections_json_validated")

categories = {
    2: "Container",
    3: "Dixie",
    4: "Steiger",
}

In [ ]:
buurten = read_buurten_geojson(
    buurten_file=os.path.join(dataset_folder, "gebieden_v1_buurten_openbaar.geojson")
).to_crs(epsg=RD_EPSG)

In [ ]:
with open(coordinates_metadata_file, "r") as f:
    json_content = json.load(f)

data: dict[str, List] = {
    "image_file_name": [],
    "geometry": [],
}

for frame in json_content["frames"]:
    data["image_file_name"].append(frame["image_file_name"])
    data["geometry"].append(
        sg.Point(frame["gps_data"]["longitude"], frame["gps_data"]["latitude"])
    )

coordinates_metadata = (
    gpd.GeoDataFrame(data=data)
    .set_crs(epsg=LAT_LON_EPSG)
    .set_index("image_file_name")
)

In [ ]:
_annotations_clustered = (
    pd.read_csv(annotations_clustered_file, delimiter=";", index_col="id")
)
annotations_clustered = gpd.GeoDataFrame(
    data=_annotations_clustered,
    geometry=_annotations_clustered["image_file_name"].map(coordinates_metadata["geometry"]),
    crs=coordinates_metadata.crs
)
annotations_clustered["category_name"] = annotations_clustered["category_id"].map(categories)

In [ ]:
detections = read_annotations_folder(
    folder_path=detections_folder,
    categories=categories.keys()
)
detections["image_name"] = detections["image_name"] + ".jpg"
detections.rename(columns={"image_name": "image_file_name", "category": "category_id"}, inplace=True)
detections["category_name"] = detections["category_id"].map(categories)
detections["geometry"] = detections["image_file_name"].map(coordinates_metadata["geometry"])
detections = detections.set_geometry("geometry").set_crs(coordinates_metadata.crs)


conf_thresholds = {
    2: 0.82,
    3: 0.83,
    4: 0.82,
}
bbox_size_threshold = {
    2: 0.008,
    3: 0.004,
    4: 0.1,
}

def filter_detection(detection_row: pd.Series) -> bool:
    class_id = detection_row["category_id"]
    area = detection_row["bbox"].area
    return (
        (detection_row["confidence"] >= conf_thresholds[class_id]) 
        and (area >= bbox_size_threshold[class_id])
    )

detections["filtered"] = detections.apply(filter_detection, axis=1)

In [ ]:
detections_validated = read_annotations_folder(
    folder_path=detections_validated_folder,
    file_type=".json",
    categories=categories.keys()
)
detections_validated.rename(columns={"image_name": "image_file_name", "category": "category_id"}, inplace=True)
detections_validated["category_name"] = detections_validated["category_id"].map(categories)
detections_validated["geometry"] = detections_validated["image_file_name"].map(coordinates_metadata["geometry"])
detections_validated = detections_validated.set_geometry("geometry").set_crs(coordinates_metadata.crs)

In [ ]:
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(15, 10))

annotations_clustered.to_crs(epsg=RD_EPSG).plot(ax=axes[0], column="category_name", cmap="tab20", categorical=True, legend=True, zorder=2)
detections_validated.to_crs(epsg=RD_EPSG).plot(ax=axes[1], column="category_name", cmap="tab20", categorical=True, legend=True, zorder=2)

add_neighborhoods_to_plot(axes, buurten)

axes[0].set_title("Manually labelled")
axes[1].set_title("Validated detections")

plt.show()

In [ ]:
conf = 0.7
_det_cat_conf: gpd.GeoDataFrame = detections[detections["confidence"]>=conf]
_det_filtered: gpd.GeoDataFrame = detections[detections["filtered"]]

fig, axes = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(20, 5))

detections_validated.to_crs(epsg=RD_EPSG).plot(ax=axes[0], column="category_name", cmap="tab20", legend=True, categorical=True, zorder=2)
_det_cat_conf.to_crs(epsg=RD_EPSG).plot(ax=axes[1], column="category_name", cmap="tab20", legend=True, categorical=True, zorder=2)
_det_filtered.to_crs(epsg=RD_EPSG).plot(ax=axes[2], column="category_name", cmap="tab20", legend=True, categorical=True, zorder=2)

add_neighborhoods_to_plot(axes, buurten)

axes[0].set_title("Validated detections")
axes[1].set_title(f"Detections @{conf:.1f} conf")
axes[2].set_title(f"Detections filtered")

plt.show()

In [ ]:
cat_id = 4
sample_frac = 0.6

cat_annotations: gpd.GeoDataFrame = annotations_clustered[annotations_clustered["category_id"]==cat_id]
n_clusters = len(cat_annotations["cluster_id"].unique())
print(f"Manual labelling: {n_clusters} clusters for category {cat_id}")

cat_annotations = cluster_and_select_images(
    gdf=cat_annotations, 
    distance=10
)
n_clusters = len(cat_annotations["tracking_id"].unique())
print(f"DBSCAN (manual labelling): {n_clusters} clusters for category {cat_id}")

cat_annotations_sampled = cat_annotations.sample(
    frac=sample_frac,
    axis="index"
)
cat_annotations_sampled = cluster_and_select_images(
    gdf=cat_annotations_sampled, 
    distance=10
)
n_clusters = len(cat_annotations_sampled["tracking_id"].unique())
print(f"DBSCAN (manual labelling, sample {sample_frac:.1f}): {n_clusters} clusters for category {cat_id}")


fig, axes = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(20, 4.5))

cat_annotations.to_crs(epsg=RD_EPSG).plot(ax=axes[0], column="cluster_id", cmap="tab20", categorical=True, zorder=2)
cat_annotations.to_crs(epsg=RD_EPSG).plot(ax=axes[1], column="tracking_id", cmap="tab20", categorical=True, zorder=2)
cat_annotations_sampled.to_crs(epsg=RD_EPSG).plot(ax=axes[2], column="tracking_id", cmap="tab20", categorical=True, zorder=2)

add_neighborhoods_to_plot(axes, buurten)

axes[0].set_title("Manual clustering")
axes[1].set_title("DBSCAN (manual labelling)")
axes[2].set_title(f"DBSCAN (manual labelling, sample {sample_frac:.1f})")

fig.suptitle(f"Clusters for {categories[cat_id]}")

plt.show()

In [ ]:
fig.savefig(
    fname=os.path.join(dataset_folder, f"Manual_labelling_DBSCAN_{categories[cat_id]}.png"),
    dpi=150,
    bbox_inches="tight"
)

In [ ]:
dist = [5, 10, 20, 30]
_dist_cat_annotations = []

for d in dist:
    _tmp = cluster_and_select_images(
        gdf=cat_annotations.to_crs(epsg=RD_EPSG), 
        distance=d
    )
    n_clusters = len(_tmp["tracking_id"].unique())
    print(f"DBSCAN (distance={d}): {n_clusters} clusters for category {cat_id}")
    _dist_cat_annotations.append(_tmp)

fig, axes = plt.subplots(2, 3, sharex=True, sharey=True, figsize=(20, 8))

cat_annotations.to_crs(epsg=RD_EPSG).plot(ax=axes[0, 0], column="cluster_id", cmap="tab20", categorical=True, zorder=2)
axes[0, 0].set_title("Manual clustering")

for idx, ax in enumerate(axes[:, 1:].reshape(-1)):
    _dist_cat_annotations[idx].plot(ax=ax, column="tracking_id", cmap="tab20", categorical=True, zorder=2)
    ax.set_title(f"DBSCAN (distance={dist[idx]})")

add_neighborhoods_to_plot(axes[0], buurten)
add_neighborhoods_to_plot(axes[1], buurten)

axes[1, 0].set_visible(False)

fig.suptitle(f"Clusters for {categories[cat_id]}, manual labelling")

plt.show()

In [ ]:
fig.savefig(
    fname=os.path.join(dataset_folder, f"DBSCAN_dist_{categories[cat_id]}.png"),
    dpi=150,
    bbox_inches="tight"
)

In [ ]:
_det_cat_conf: gpd.GeoDataFrame = detections[
    (detections["category_id"]==cat_id) & (detections["confidence"]>=conf)
]
_det_cat_conf = cluster_and_select_images(
    gdf=_det_cat_conf, 
    distance=10
)
n_clusters = len(_det_cat_conf["tracking_id"].unique())
print(f"Detections @{conf:.1f} conf: {n_clusters} clusters for category {cat_id}")

_det_cat_filtered: gpd.GeoDataFrame = detections[
    (detections["category_id"]==cat_id) & (detections["filtered"])
]
_det_cat_filtered = cluster_and_select_images(
    gdf=_det_cat_filtered, 
    distance=10
)
n_clusters = len(_det_cat_filtered["tracking_id"].unique())
print(f"Detections filtered: {n_clusters} clusters for category {cat_id}")


fig, axes = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(20, 4.9))

cat_annotations.to_crs(epsg=RD_EPSG).plot(ax=axes[0], column="cluster_id", cmap="tab20", categorical=True, zorder=2)
_det_cat_conf.to_crs(epsg=RD_EPSG).plot(ax=axes[1], column="tracking_id", cmap="tab20", categorical=True, zorder=2)
_det_cat_filtered.to_crs(epsg=RD_EPSG).plot(ax=axes[2], column="tracking_id", cmap="tab20", categorical=True, zorder=2)

add_neighborhoods_to_plot(axes, buurten)

axes[0].set_title("Manual clustering")
axes[1].set_title(f"DBSCAN (detections @{conf:.1f} conf)")
axes[2].set_title("DBSCAN (detections filtered)")

fig.suptitle(f"Clusters for {categories[cat_id]}")

plt.show()

In [ ]:
fig.savefig(
    fname=os.path.join(dataset_folder, f"Detections_DBSCAN_{categories[cat_id]}.png"),
    dpi=150,
    bbox_inches="tight"
)